# 00 · Setup & Environment Check

**Purpose.** Confirm Colab, Drive, libraries and every raw dataset are in place before any analysis.

| | |
|---|---|
| **Reads** | `data/raw/*`, `configs/baseline.yaml` |
| **Writes** | nothing (optional: archives v1 outputs) |
| **Run time (Colab T4)** | ~1 min |
| **Next notebook** | 01 · Data Audit |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

### Project layout
```
DermaLens_AI/                  <- this Drive folder is also the Git repository
├── configs/baseline.yaml      every setting (image size, learning rate, ...)
├── src/dermalens/             shared code, imported by every notebook
├── notebooks/                 00 → 12, run in order
├── data/raw/                  original downloads, never modified
├── data/processed/            clean tables built from raw
├── data/splits/               train / val / test CSVs + class_info.json
├── data/cache/                small resized copies of the images
├── models/                    checkpoints + the web-app bundle
├── reports/                   audit CSVs, figures, results
└── app/                       FastAPI web prototype
```

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

## 1. Hardware and library versions
Training needs a GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import platform
import importlib

print("Python", platform.python_version())
for package in ["numpy", "pandas", "sklearn", "PIL", "torch", "torchvision", "timm", "imagehash"]:
    try:
        module = importlib.import_module(package)
        print(f"{package:<12} {getattr(module, '__version__', 'installed')}")
    except ImportError:
        print(f"{package:<12} MISSING  -> run: pip install -r requirements.txt")

import torch
print("\nGPU available:", torch.cuda.is_available(),
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "- training will be slow")

## 2. Are all raw datasets where the code expects them?

In [ ]:
import pandas as pd

# Each source dataset and the one metadata file the pipeline depends on.
expected = {
    "dermaconin": paths.raw / "dermaconin" / "METADATA" / "Skin_Metadata-1.csv",
    "skindisnet": paths.raw / "skindisnet" / "skindisnet2" / "SkinDisNet_Metadata.csv",
    "mcsi":       paths.raw / "mcsi" / "extracted" / "metadata.csv",
    "scin":       paths.raw / "scin" / "dataset_scin_cases.csv",
}
check = pd.DataFrame([
    {"dataset": name, "metadata_file": str(path.relative_to(paths.raw)), "found": path.is_file()}
    for name, path in expected.items()
])
display(check)
assert check.loc[check["dataset"] != "scin", "found"].all(), "A required dataset is missing: fix before continuing."

## 3. One-time clean-up of version-1 outputs (optional)

Version 1 left an older split set in `data/processed/` (`train.csv`, `val.csv`, `test.csv`,
`class_map.json`). They are **moved, not deleted**, to `data/processed/_archive_v1/`, so nobody
trains on the wrong split by mistake. Set the flag to `True` once, after the team agrees.

In [ ]:
ARCHIVE_V1_OUTPUTS = False

stale_files = ["train.csv", "val.csv", "test.csv", "class_map.json",
               "dermaconin_manifest.csv", "skindisnet_manifest.csv"]
archive_dir = paths.processed / "_archive_v1"
for name in stale_files:
    source = paths.processed / name
    if not source.exists():
        continue
    if ARCHIVE_V1_OUTPUTS:
        archive_dir.mkdir(exist_ok=True)
        source.rename(archive_dir / name)
        print("moved   ", name, "->", archive_dir.name)
    else:
        print("would move", name, "(set ARCHIVE_V1_OUTPUTS = True)")

**Done.** If every check above passed, continue with notebook 01.